In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os


In [11]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="flan_t5_yes_no_prompt_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()


---[ TableVault Record ]---
---[ TableVault Record ]---



In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [3]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

yes_ids = tokenizer.encode("yes", add_special_tokens=False)
no_ids = tokenizer.encode("no", add_special_tokens=False)

print("model:", model_name)
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("yes ids:", yes_ids, "decoded:", tokenizer.decode(yes_ids))
print("no ids:", no_ids, "decoded:", tokenizer.decode(no_ids))

assert len(yes_ids) == 1, f"Expected single token for 'yes', got {yes_ids}"
assert len(no_ids) == 1, f"Expected single token for 'no', got {no_ids}"

yes_token_id = yes_ids[0]
no_token_id = no_ids[0]
start_token_id = model.config.decoder_start_token_id
if start_token_id is None:
    start_token_id = tokenizer.pad_token_id

print("yes_token_id:", yes_token_id)
print("no_token_id:", no_token_id)
print("start_token_id:", start_token_id)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model: google/flan-t5-small
decoder_start_token_id: 0
yes ids: [4273] decoded: yes
no ids: [150] decoded: no
yes_token_id: 4273
no_token_id: 150
start_token_id: 0


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

prompts = [
    f"Do these two sentences have the same meaning? Answer yes or no.\nSentence 1: {a}\nSentence 2: {b}"
    for a, b in zip(sent1, sent2)
]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print("sample_prompt:\n", prompts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
sample_prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [5]:
batch_size = 32
preds = []
yes_logits_all = []
no_logits_all = []


with torch.no_grad():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        decoder_input_ids = torch.full(
            (len(batch_prompts), 1),
            start_token_id,
            dtype=torch.long,
            device=device,
        )

        outputs = model(**enc, decoder_input_ids=decoder_input_ids)
        first_step_logits = outputs.logits[:, 0, :]

        yes_logits = first_step_logits[:, yes_token_id]
        no_logits = first_step_logits[:, no_token_id]

        batch_preds = (yes_logits > no_logits).long().cpu().numpy()
        preds.extend(batch_preds.tolist())
        yes_logits_all.extend(yes_logits.detach().cpu().numpy().tolist())
        no_logits_all.extend(no_logits.detach().cpu().numpy().tolist())

y_pred = np.array(preds)
yes_logits_all = np.array(yes_logits_all)
no_logits_all = np.array(no_logits_all)
print("done")


  0%|          | 0/13 [00:00<?, ?it/s]

done


In [ ]:
vault.create_record_list("flan_yes_no_prediction_and_logits", column_names=["prediction", "yes_logit", "no_logit"])
for i in range(len(y_pred)):
    vault.append_record(
        "flan_yes_no_prediction_and_logits",
        {"prediction": y_pred[i],
         "yes_logit": yes_logits_all[i],
         "no_logit": no_logits_all[i]
        },
        input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
    )

description = "Per-example inference results from running google/flan-t5-small on the GLUE MRPC validation set with a yes/no paraphrase prompt. Each record corresponds to one input pair from glue_mrpc_validation and stores the model\u2019s binary prediction and first-decoder-step scores for the answer tokens. Columns: prediction (0/1, where 1 means predicted \u201cyes\u201d/paraphrase and 0 means predicted \u201cno\u201d/not paraphrase), yes_logit (logit for the token \u201cyes\u201d), and no_logit (logit for the token \u201cno\u201d). This dataset serves as the intermediate output of the workflow, enabling per-example analysis, error inspection, and computation of aggregate evaluation metrics in the summary dataset."
embedding = get_embeddings(description)
vault.create_description("flan_yes_no_prediction_and_logits", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "model predictions and logits", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prompting": "yes/no prompt", "prediction_target": "same meaning", "input_format": "sentence pair", "output_columns": "prediction, yes_logit, no_logit", "label_space": "0=not_paraphrase, 1=paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_yes_no_prediction_and_logits", cat, embedding, prop)

In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], zero_division=0))


{'accuracy': 0.6838235294117647, 'f1': 0.7393939393939394}
                precision    recall  f1-score   support

not_paraphrase       0.50      0.74      0.60       129
    paraphrase       0.85      0.66      0.74       279

      accuracy                           0.68       408
     macro avg       0.67      0.70      0.67       408
  weighted avg       0.74      0.68      0.69       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("idx:", i)
    print("prompt:\n", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", "yes" if int(y_pred[i]) == 1 else "no")
    print("yes_logit:", float(yes_logits_all[i]))
    print("no_logit:", float(no_logits_all[i]))


idx: 0
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: yes
yes_logit: -1.8457680940628052
no_logit: -2.246473550796509
idx: 1
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
Sentence 2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: no
yes_logit: -4.14354133605957
no_logit: -2.2493021488189697
idx: 2
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
Sent

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("prompt:\n", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("yes_logit:", float(yes_logits_all[i]))
    print("no_logit:", float(no_logits_all[i]))


num_errors: 129
idx: 3
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
Sentence 2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
true: 1 pred: 0
yes_logit: -3.1410040855407715
no_logit: -2.7465322017669678
idx: 7
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
Sentence 2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C # and VisualBasic .Net.
true: 1 pred: 0
yes_logit: -2.598281145095825
no_logit: -2.3498666286468506
idx: 9
prompt:
 Do these two sentences have the same meaning? Answer yes or no.
Sentence 1: The results appear in the January issue of 

In [9]:
vault.create_record_list("flan_t5_yes_no_prompt_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("flan_t5_yes_no_prompt_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "flan_yes_no_prediction_and_logits": [0, len(ds)]
                    })

summary

description = "This dataset stores the aggregate evaluation results for the FLAN-T5 yes/no prompting experiment on the GLUE MRPC validation set. It contains summary metrics computed by comparing the model\u2019s binary predictions against the ground-truth MRPC labels after prompting each sentence pair with a yes/no paraphrase question.\n\nStructure: a record list with three fields: accuracy (float), f1 (float), and classification_report (string). In this workflow, it is populated with a single summary record covering the full validation split.\n\nRole in the workflow: it serves as the experiment-level evaluation artifact, summarizing overall model performance after the per-example predictions and logits have been generated and stored in flan_yes_no_prediction_and_logits. It is linked to both the full glue_mrpc_validation input slice and the full prediction/logit output slice for traceability."
embedding = get_embeddings(description)
vault.create_description("flan_t5_yes_no_prompt_mrpc_summary", description, embedding)

properties = {"artifact_type": "evaluation summary", "task": "paraphrase detection", "source": "glue/mrpc", "benchmark": "GLUE", "dataset": "MRPC", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prompt_format": "yes/no question answering", "label_space": "binary", "metrics": "accuracy,f1,classification_report", "input_type": "sentence pair", "prediction_source": "flan_yes_no_prediction_and_logits", "process": "flan_t5_yes_no_prompt_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_yes_no_prompt_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'google/flan-t5-small',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6838235294117647,
 'f1': 0.7393939393939394}

In [ ]:
description = "This notebook evaluates a prompt-based paraphrase detection workflow on the GLUE MRPC validation set using google/flan-t5-small. It treats the task as binary yes/no generation: for each pair of sentences, it builds a prompt asking whether the two sentences have the same meaning, then runs FLAN-T5 and compares the first-step decoder logits for the tokens \u201cyes\u201d and \u201cno\u201d to produce a prediction. The notebook loads the MRPC validation data from TableVault, performs batched inference with PyTorch, collects per-example predictions and yes/no logits, and computes standard classification metrics including accuracy, F1, and a classification report. It also inspects sample predictions and errors, and writes both the detailed prediction records and the evaluation summary back to TableVault, along with embedding-based descriptions and metadata for later discovery and documentation." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("flan_t5_yes_no_prompt_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary text classification", "model": "google/flan-t5-small", "model_family": "Flan-T5", "dataset": "glue/mrpc", "dataset_split": "validation", "prompting": "yes/no prompt-based inference", "inference_method": "first-token logit comparison", "labels": "yes/no", "framework": "transformers", "library": "huggingface transformers", "metrics": "accuracy, f1-score, classification_report", "outputs": "predictions, yes_logit, no_logit, evaluation summary", "tracking": "tablevault", "embedding_model": "text-embedding-3-large", "hardware": "torch mps/cpu", "notebook_type": "model evaluation"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_yes_no_prompt_mrpc", cat, embedding, prop)